# Phase 1 — Build the 512 px cache

Decodes every fundus image once instead of every epoch. This is the difference
between a ~4 hour training run and a ~1.5 hour one, and therefore between a feasible
thesis and an infeasible one.

## Notebook settings (right-hand panel)

| Setting | Value | Why it matters |
|---|---|---|
| Accelerator | **None** | This job is pure CPU. Attaching a GPU burns a third of your weekly quota on JPEG decoding. |
| Persistence | **Files only** | `build_cache.py` resumes. Persistence lets a 10-hour build span several sittings with zero rework. |
| Internet | On (git clone only) | cv2 and numpy are preinstalled; nothing else is needed. |
| Environment | **Pin to original environment** | Kaggle updates its base image. Over 20 weeks that will silently change something. |

## Inputs

EyePACS, DDR, IDRiD now. APTOS and Messidor-2 can come later — they aren't touched
until Phase 6.

## How to run this

Work through **step 1 interactively and look at the contact sheet**. Only once the
crops look right, run the rest with *Save & Run All (Commit)* so it continues
unattended — interactive sessions die when your browser idles.

## Exit condition

`cache_report.json` for each dataset with a crop fallback rate ≤ 0.005, and the cache
saved as a Kaggle dataset. **Never rebuild it after Phase 2** — every downstream
number assumes a fixed cache.

## 1 · Pull the code

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"
print("repo ready at", REPO_DIR)

In [ ]:
from pathlib import Path
import os, re, json

INPUT = Path("/kaggle/input")

def _norm(s):
    return s.lower().replace("-", "").replace("_", "").replace("%20", "")

def dataset_roots(max_depth=3):
    """Candidate dataset directories, shallowest first.

    Kaggle does not always mount datasets as direct children of /kaggle/input --
    they can sit under competitions/ and datasets/ wrappers. Breadth-first so a
    shallower match always wins over a nested subfolder of the same name.
    """
    level, out = [INPUT], []
    for _ in range(max_depth):
        nxt = []
        for d in level:
            try:
                children = sorted(c for c in d.iterdir() if c.is_dir())
            except OSError:
                continue
            out.extend(children)
            nxt.extend(children)
        level = nxt
    return out

EXCLUDE_ROOTS = []      # set to [CACHE] once the cache is located

def _under(path, root):
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False

def find_mount(*keywords, required=True):
    """Locate a mounted dataset by keyword, so a renamed mirror doesn't break the notebook.

    Anything under EXCLUDE_ROOTS is skipped. The cache contains directories named
    ddr, eyepacs, idrid, aptos and messidor2 -- exactly the keywords searched for --
    so without this a raw-dataset lookup can land inside the cache instead.
    """
    for d in dataset_roots():
        if any(_under(d, r) for r in EXCLUDE_ROOTS):
            continue
        if all(_norm(k) in _norm(d.name) for k in keywords):
            return d
    if required:
        print(f"  !! NOT MOUNTED: {' + '.join(keywords)}")
        print("     Use 'Add Input' in the right-hand panel. Candidate dirs seen:")
        for d in dataset_roots(2)[:25]:
            print(f"       - {d.relative_to(INPUT)}")
    return None

def find_any(*keyword_sets, label="", required=True):
    """Try several keyword spellings. Mirrors title themselves inconsistently:
    IDRiD ships as 'idrid-dataset' or 'indian-diabetic-retinopathy-image-dataset'."""
    for kws in keyword_sets:
        hit = find_mount(*kws, required=False)
        if hit:
            return hit
    if required:
        print(f"  !! NOT MOUNTED: {label or keyword_sets[0]}")
        print("     Currently mounted:")
        for d in sorted(INPUT.iterdir()):
            print(f"       - {d.name}")
    return None

def match_channel(dirname):
    """Map a mask directory name to a lesion channel.

    DDR uses MA/HE/EX/SE; IDRiD uses '1. Microaneurysms', '2. Haemorrhages',
    '3. Hard Exudates', '4. Soft Exudates', '5. Optic Disc'. Match on meaning so
    one function covers both.
    """
    n = re.sub(r"^\d+\.\s*", "", dirname.lower().strip())
    n = n.replace("%20", " ")
    if n == "ma" or "microaneurysm" in n:
        return "microaneurysm"
    if n == "he" or "haemorrhage" in n or "hemorrhage" in n:
        return "haemorrhage"
    if n == "ex" or ("hard" in n and "exudate" in n):
        return "hard_exudate"
    if n == "se" or ("soft" in n and "exudate" in n) or "cotton" in n:
        return "soft_exudate"
    if n == "od" or "optic disc" in n:
        return "optic_disc"
    return None

MASK_EXT = {".tif", ".tiff", ".png", ".gif", ".bmp", ".jpg", ".jpeg"}
LESION4 = ["microaneurysm", "haemorrhage", "hard_exudate", "soft_exudate"]

def scan_mask_dirs(root, require=""):
    """Find mask directories under root, keyed by lesion channel."""
    found = {}
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        if require and require not in str(d).lower():
            continue
        channel = match_channel(d.name)
        if not channel:
            continue
        files = [f for f in d.rglob("*") if f.suffix.lower() in MASK_EXT]
        if files:
            found.setdefault(channel, []).append((str(d), len(files)))
    return found

def du(path, cap=40000):
    """Rough size + file count, capped so it stays fast on huge mounts."""
    total = n = 0
    for i, f in enumerate(Path(path).rglob("*")):
        if i > cap:
            return total, n, True
        if f.is_file():
            total += f.stat().st_size
            n += 1
    return total, n, False

print("helpers ready")

import shlex

CACHE = Path("/kaggle/working/cache512")
SCRIPT = REPO_DIR / "scripts/build_cache.py"

def build(source_root, dataset, masks=None, extra=""):
    """Assemble a quoted build_cache.py command.

    IDRiD directory names contain spaces ("A. Segmentation", "1. Original
    Images"), so every path must be shell-quoted or the command splits apart.
    """
    q = shlex.quote
    cmd = (f"python {q(str(SCRIPT))} --source-root {q(str(source_root))} "
           f"--dataset {dataset} --output-root {q(str(CACHE))}")
    for channel, d in (masks or {}).items():
        cmd += f" --mask {q(f'{channel}={d}')}"
    return cmd + (f" {extra}" if extra else "")

print("cache ->", CACHE)

---
## 2 · Smoke test — 200 images, then look

Never launch a ten-hour build on an unverified crop. This takes about a minute.

In [ ]:
eyepacs = find_mount("eyepacs")
cmd = build(eyepacs, "EyePACS", extra="--limit 200 --contact-sheet 100")
print(cmd)
!{cmd}

In [ ]:
from IPython.display import Image, display
display(Image(str(CACHE / "eyepacs/contact_sheet.jpg")))

### ⚠️ Stop here and actually look at that sheet

**Good:** each retina fills its tile, roughly circular, tightly framed, centred. A
little black in the corners is expected — `--fit pad` keeps the full field on purpose,
because M3's quadrant rules need the periphery.

**Bad, and what it means:**

| What you see | Cause | Fix |
|---|---|---|
| Wide black bars left/right | Crop didn't fire; surround noise defeated the threshold | Raise `--tol-scale` to 0.15–0.20 and re-run |
| Retina cut off at the edges | Crop too aggressive | Lower `--tol-scale` to 0.05 |
| Mostly-black or blank tiles | Genuinely ungradable images | Expected in EyePACS — quantify, don't fix |
| Washed-out, flat contrast | CLAHE too strong for this source | Lower `--clahe-clip` to 1.5, or `--no-clahe` |

Also check the printed **crop fallback rate**. A0's gate is ≤ 0.005; the script flags
it when exceeded. A high rate on the smoke sample is worth understanding *now*.

Re-run the cell above until the sheet looks right. Only then continue.

---
## 3 · Full EyePACS build

Around 45–90 minutes depending on how many images the mirror carries. Resumable — if
the session dies, just re-run this cell. **Never pass `--overwrite`**, that throws
away completed work.

In [ ]:
cmd = build(eyepacs, "EyePACS", extra="--contact-sheet 100")
!{cmd}

---
## 4 · DDR — grading images and the lesion masks

Two passes. The first caches the grading images; the second caches the segmentation
subset **with its masks**, which is what M2 trains on.

Mask paths come from `verification_log.json` written by `00_verify_inputs.ipynb`, so
this adapts to whatever layout your DDR mirror actually uses.

In [ ]:
ddr = find_mount("ddr")
grading = next((p for p in ddr.rglob("*")
                if p.is_dir() and "grading" in p.name.lower()), ddr)
print("DDR grading images:", grading)

cmd = build(grading, "DDR", extra="--contact-sheet 60")
!{cmd}

In [ ]:
# Phase 0's verification_log.json is used when present, but /kaggle/working is
# per-notebook, so this notebook re-discovers everything itself rather than
# depending on a file it will usually not find.
log = Path("/kaggle/working/verification_log.json")
saved = json.loads(log.read_text()) if log.exists() else {}

mask_dirs = saved.get("ddr_mask_dirs") or {}
if mask_dirs:
    print("DDR mask dirs: loaded from verification_log.json")
else:
    mask_dirs = {c: sorted(d for d, _ in v)
                 for c, v in scan_mask_dirs(ddr, require="segmentation").items()}
    print("DDR mask dirs: re-discovered in this notebook")

seg_images = saved.get("ddr_seg_image_dirs") or sorted(
    str(p) for p in ddr.rglob("*")
    if p.is_dir() and "segmentation" in str(p).lower()
    and p.name.lower() in {"image", "images"})

print("mask channels:", sorted(mask_dirs))
print("segmentation image dirs:", seg_images)
assert mask_dirs, "DDR lesion masks not found - check Q1 in 00_verify_inputs.ipynb"
assert seg_images, "DDR segmentation images not found - check Q1"

In [ ]:
# One pass per split directory (train/valid/test), each with its own mask folders.
for img_dir in map(Path, seg_images):
    split = img_dir.parent.name
    chosen = {}
    for channel, dirs in mask_dirs.items():
        match = [d for d in dirs if f"/{split}/" in d + "/"]
        if match:
            chosen[channel] = match[0]
    if not chosen:
        print(f"skip {split}: no matching mask dirs")
        continue
    print(f"\n=== DDR segmentation split: {split} ===")
    cmd = build(img_dir, "DDR", masks=chosen, extra="--contact-sheet 0")
    !{cmd}

---
## 5 · IDRiD — lesion masks and geometry

Small (516 images) and quick. The coordinate CSVs aren't cached — `prepare_manifest.py`
reads them directly in Phase 2.

In [ ]:
idrid = find_any(("idrid",), ("indian", "diabetic", "retinopathy"), label="IDRiD")

# Part A pairs original images with per-lesion mask folders. Match each image
# directory to the mask dirs of the same split (Training / Testing Set).
idrid_masks, idrid_images = saved.get("idrid_mask_dirs") or {}, saved.get("idrid_seg_image_dirs") or []

if idrid and not idrid_masks:
    idrid_masks = {c: sorted(d for d, _ in v)
                   for c, v in scan_mask_dirs(idrid).items()}
if idrid and not idrid_images:
    # Part A images live in "1. Original Images/<split>", so the directory NAME
    # is the split; "original" has to be matched against the whole path.
    idrid_images = sorted(
        str(d) for d in idrid.rglob("*")
        if d.is_dir() and "original" in str(d).lower()
        and "segmentation" in str(d).lower()
        and any(f.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif"}
                for f in d.iterdir() if f.is_file()))

print("IDRiD mask channels :", sorted(idrid_masks))
print("IDRiD Part A images :", len(idrid_images), "dir(s)")

def split_key(path):
    low = path.lower()
    if "train" in low:
        return "train"
    return "test" if "test" in low else None

if idrid_images and idrid_masks:
    for img_dir in idrid_images:
        key = split_key(img_dir)
        chosen = {}
        for channel, dirs in idrid_masks.items():
            if channel == "optic_disc":
                continue                      # geometry, not a lesion channel
            match = [d for d in dirs if split_key(d) == key]
            if match:
                chosen[channel] = match[0]
        print(f"\n=== IDRiD Part A ({key}): {len(chosen)} mask channels ===")
        cmd = build(img_dir, "IDRiD", masks=chosen, extra="--contact-sheet 40")
        !{cmd}
elif idrid:
    print("!! No Part A image/mask pairing found - caching IMAGES ONLY.")
    print("!! C2 will then train on DDR masks alone. If you mounted the full")
    print("!! three-part IDRiD, re-check Q4 in 00_verify_inputs.ipynb.")
    cmd = build(idrid, "IDRiD", extra="--contact-sheet 60")
    !{cmd}

---
## 6 · APTOS and Messidor-2 — *optional now*

These are the **locked external sets**. Caching them is harmless — it's a decode, not
an evaluation — but nothing may read them until after the Phase 5 freeze
(`docs/02_research_protocol.md` Rule 1).

Skip this cell if you'd rather not have them on disk at all until Phase 6.

In [ ]:
for keywords, name in [(("aptos",), "APTOS"), (("messidor2preprocess",), "Messidor2")]:
    mount = find_mount(*keywords, required=False)
    if mount is None:
        print(f"{name}: not mounted, skipping")
        continue
    src = next((p for p in mount.rglob("*") if p.is_dir()
                and "train" in p.name.lower() and "image" in p.name.lower()), mount)
    print(f"\n=== {name} from {src} ===")
    cmd = build(src, name, extra="--contact-sheet 60")
    !{cmd}

---
## 7 · Experiment A0 — the gate

A0 passes when counts reconcile, the crop fallback rate is ≤ 0.005 everywhere, and the
contact sheets look right. Record the result in `docs/04_experiment_register.md`.

In [ ]:
rows = []
for report in sorted(CACHE.rglob("cache_report.json")):
    r = json.loads(report.read_text())
    rows.append(r)

print(f"{'dataset':<12}{'found':>8}{'cached':>8}{'failed':>8}{'fallback':>11}{'MiB':>9}{'runs':>6}  gate")
print("-" * 76)
a0 = True
for r in rows:
    found, cached = r["counts"]["found"], r["cached"]
    failed, rate = r["counts"]["failed"], r["crop"]["fallback_rate"]
    mib = r["output"]["total_mib"]
    # A0: everything cached, crop fallback under 0.005, failures under 0.1%
    ok = (cached >= found - max(1, 0.001 * found)
          and rate is not None and rate <= 0.005)
    a0 &= ok
    shown = f"{rate:.4f}" if rate is not None else "    n/a"
    print(f"{r['dataset']:<12}{found:>8}{cached:>8}{failed:>8}{shown:>11}"
          f"{mib:>9.0f}{r['runs']:>6}  {'PASS' if ok else 'FAIL'}")
    for channel, stat in r["masks"].items():
        if stat["written"]:
            print(f"    mask {channel:<16} written={stat['written']}")
    if r["failures"]:
        print(f"    first failure: {r['failures'][0]['path']} - {r['failures'][0]['error']}")

total = sum(r["output"]["total_mib"] for r in rows)
print("-" * 76)
print(f"total cache: {total:.0f} MiB")
print("\nA0", "PASS - proceed to Phase 2" if a0 else "FAIL - investigate before continuing")
print("Now eyeball each contact sheet below, then record A0 in docs/04_experiment_register.md.")

In [ ]:
from IPython.display import Image, display
for sheet in sorted(CACHE.rglob("contact_sheet.jpg")):
    print(sheet.parent.name)
    display(Image(str(sheet)))

---
## 8 · Save the cache

**Save Version → Save & Run All (Commit).** When it finishes, open the notebook's
Output tab and use *New Dataset* to publish `/kaggle/working/cache512` as a private
dataset named **`verify-dr-cache-512`**.

Every notebook from Phase 2 on takes that dataset as input and never touches the raw
sources again.

> **Freeze it.** Rebuilding the cache after Phase 2 invalidates every split,
> checkpoint and number that came from it. If you genuinely must rebuild, treat it as
> a protocol deviation and record it in `preregistration/PREREGISTRATION.md`.

**Next:** `02_manifests.ipynb` — blocked until `prepare_manifest.py` and
`build_variants.py` are written.